#데이터 경로 설정

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

In [2]:
pip install catboost

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
df_train = pd.read_csv("train.csv") # 학습용 데이터
df_test = pd.read_csv("submission.csv") # 테스트 데이터(제출파일의 데이터)

In [4]:
df_train.head(5)

,bant_submit,customer_country,business_unit,com_reg_ver_win_rate,customer_idx,customer_type,enterprise,historical_existing_cnt,id_strategic_ver,it_strategic_ver,...,response_corporate,expected_timeline,ver_cus,ver_pro,ver_win_rate_x,ver_win_ratio_per_bu,business_area,business_subarea,lead_owner,is_converted
0,1.0,/Quezon City/Philippines,AS,0.066667,32160,End-Customer,Enterprise,NaN,NaN,NaN,...,LGEPH,less than 3 months,1,0,0.003079,0.026846,corporate / office,Engineering,0,True
1,1.0,/PH-00/Philippines,AS,0.066667,23122,End-Customer,Enterprise,12.0,NaN,NaN,...,LGEPH,less than 3 months,1,0,0.003079,0.026846,corporate / office,Advertising,1,True
2,1.0,/Kolkata /India,AS,0.088889,1755,End-Customer,Enterprise,144.0,NaN,NaN,...,LGEIL,less than 3 months,1,0,0.003079,0.026846,corporate / office,Construction,2,True
3,1.0,/Bhubaneswar/India,AS,0.088889,4919,End-Customer,Enterprise,NaN,NaN,NaN,...,LGEIL,less than 3 months,1,0,0.003079,0.026846,corporate / office,IT/Software,3,True
4,1.0,/Hyderabad/India,AS,0.088889,17126,Specifier/ Influencer,Enterprise,NaN,NaN,NaN,...,LGEIL,less than 3 months,0,0,0.003079,0.026846,corporate / office,NaN,4,True


In [5]:
df_test.head(5)

,id,bant_submit,customer_country,business_unit,com_reg_ver_win_rate,customer_idx,customer_type,enterprise,historical_existing_cnt,id_strategic_ver,...,response_corporate,expected_timeline,ver_cus,ver_pro,ver_win_rate_x,ver_win_ratio_per_bu,business_area,business_subarea,lead_owner,is_converted
0,19844,0.00,/ / Brazil,ID,0.073248,47466,End Customer,Enterprise,53.0,NaN,...,LGESP,NaN,1,0,0.001183,0.049840,retail,Electronics & Telco,278,False
1,9738,0.25,400 N State Of Franklin Rd Cloud IT / Johnson...,IT,NaN,5405,End Customer,SMB,NaN,NaN,...,LGEUS,NaN,0,0,0.000013,NaN,transportation,Others,437,True
2,8491,1.00,/ / U.A.E,ID,NaN,13597,Specifier/ Influencer,SMB,NaN,NaN,...,LGEGF,less than 3 months,0,0,0.000060,0.131148,hospital & health care,General Hospital,874,True
3,19895,0.50,/ Madison / United States,ID,0.118644,17204,NaN,Enterprise,NaN,NaN,...,LGEUS,more than a year,0,0,0.001183,0.049840,retail,NaN,194,False
4,10465,1.00,/ Sao Paulo / Brazil,ID,0.074949,2329,End Customer,Enterprise,2.0,1.0,...,LGESP,less than 3 months,1,1,0.003079,0.064566,corporate / office,Engineering,167,True


In [32]:
len(df_train)

59299

In [33]:
# dataframe 열 이름과 타입 출력
print(df_train.dtypes)

bant_submit                float64
customer_country            object
business_unit               object
com_reg_ver_win_rate       float64
customer_idx                 int64
customer_type               object
enterprise                  object
historical_existing_cnt    float64
id_strategic_ver           float64
it_strategic_ver           float64
idit_strategic_ver         float64
customer_job                object
lead_desc_length             int64
inquiry_type                object
product_category            object
product_subcategory         object
product_modelname           object
customer_country.1          object
customer_position           object
response_corporate          object
expected_timeline           object
ver_cus                      int64
ver_pro                      int64
ver_win_rate_x             float64
ver_win_ratio_per_bu       float64
business_area               object
business_subarea            object
lead_owner                   int64
is_converted        

비정형 데이터 열 인덱스:1 2 5 6 11 13 14 15 16 17 18 19 20 25 26

In [34]:
# 특정 열 인덱스를 리스트로 지정
selected_columns_indexes = [1] # 1, 2, 5, 6, 11, 13, 14, 15, 16, 17, 18, 19, 20, 25, 26

# 열 인덱스를 사용하여 해당 열의 이름을 찾습니다.
selected_columns_names = df_train.columns[selected_columns_indexes]

# 선택된 열의 이름에 대해서만 고유한 값의 범주를 출력
for column in selected_columns_names:
    print(f"{column}: {df_train[column].unique()}")


customer_country: ['/Quezon City/Philippines' '/PH-00/Philippines' '/Kolkata /India' ...
 '/Pisco/Peru' '/santa cruz bolivia/Peru' '/paris/France']


#비정형 데이터 중 자연어 처리해야할 부분 선별

---
index
1,5,11,13,14,15,16,17,18,20,25,26


##CatBoostClassifier 사용하기

데이터 분리

In [35]:
# 라벨링을 true:1,false:0으로 바꾸기
df_train['is_converted'] = df_train['is_converted'].apply(lambda x: 1 if x == True else 0)

In [36]:
df_train['customer_type'].fillna('others', inplace=True)

In [37]:
df_train['customer_job'].fillna('none', inplace=True)

In [38]:
df_train['inquiry_type'].fillna('others', inplace=True)

In [39]:
df_train['lead_owner'] = df_train['lead_owner'].astype(str)

In [40]:
#가공변수:고객의 국적과 담당 자사 법인명 기반의 지역 정보(대륙)이 일치 여부(index:27)
df_train['same_country'] = (df_train['customer_country'] == df_train['customer_country.1']).astype(int)
df_train['same_country'] = df_train['same_country'].map({0: '불일치', 1: '일치'})

In [41]:
#가공변수 생성으로 인해 customer_country.1 열 제거
df_train.drop('customer_country.1', axis=1, inplace=True)

In [42]:
# 독립변수와 종속변수 분리
train_x = df_train.drop(['is_converted'], axis = 1)
train_y = df_train['is_converted']

In [43]:
# lead_desc_length의 전체 평균 계산
lead_desc_length_avg = train_x['lead_desc_length'].mean()

In [44]:
train_x['historical_existing_cnt'].fillna(0, inplace=True)

In [45]:
def calculate_engagement_weight(row):
    weight = 0
    # bant_submit 값이 0.5 미만인 경우
    if row['bant_submit'] < 0.5:
        weight += 0.5
    # lead_desc_length가 평균 이상인 경우 (평균을 다시 계산하지 않고, 함수 외부에서 계산된 평균을 사용)
    if row['lead_desc_length'] > lead_desc_length_avg:
        weight += 1.5
    # historical_existing_cnt가 0인 경우
    if row['historical_existing_cnt'] == 0:
        weight += 1.0
    return weight

In [46]:
# 각 행마다 가중치 총합 계산
train_x['engagement_weight_sum'] = train_x.apply(calculate_engagement_weight, axis=1)

In [47]:
train_x.drop('bant_submit', axis=1, inplace=True)

In [48]:
train_x.drop('historical_existing_cnt', axis=1, inplace=True)

In [49]:
train_x.drop('lead_desc_length', axis=1, inplace=True)

In [50]:
train_x.head(5)

,customer_country,business_unit,com_reg_ver_win_rate,customer_idx,customer_type,enterprise,id_strategic_ver,it_strategic_ver,idit_strategic_ver,customer_job,...,expected_timeline,ver_cus,ver_pro,ver_win_rate_x,ver_win_ratio_per_bu,business_area,business_subarea,lead_owner,same_country,engagement_weight_sum
0,/Quezon City/Philippines,AS,0.066667,32160,End-Customer,Enterprise,NaN,NaN,NaN,purchasing,...,less than 3 months,1,0,0.003079,0.026846,corporate / office,Engineering,0,일치,1.0
1,/PH-00/Philippines,AS,0.066667,23122,End-Customer,Enterprise,NaN,NaN,NaN,media and communication,...,less than 3 months,1,0,0.003079,0.026846,corporate / office,Advertising,1,일치,1.5
2,/Kolkata /India,AS,0.088889,1755,End-Customer,Enterprise,NaN,NaN,NaN,engineering,...,less than 3 months,1,0,0.003079,0.026846,corporate / office,Construction,2,일치,0.0
3,/Bhubaneswar/India,AS,0.088889,4919,End-Customer,Enterprise,NaN,NaN,NaN,entrepreneurship,...,less than 3 months,1,0,0.003079,0.026846,corporate / office,IT/Software,3,일치,1.0
4,/Hyderabad/India,AS,0.088889,17126,Specifier/ Influencer,Enterprise,NaN,NaN,NaN,consulting,...,less than 3 months,0,0,0.003079,0.026846,corporate / office,NaN,4,일치,2.5


In [51]:
df_train = pd.concat([train_x, train_y], axis = 1)
df_train.head()

,customer_country,business_unit,com_reg_ver_win_rate,customer_idx,customer_type,enterprise,id_strategic_ver,it_strategic_ver,idit_strategic_ver,customer_job,...,ver_cus,ver_pro,ver_win_rate_x,ver_win_ratio_per_bu,business_area,business_subarea,lead_owner,same_country,engagement_weight_sum,is_converted
0,/Quezon City/Philippines,AS,0.066667,32160,End-Customer,Enterprise,NaN,NaN,NaN,purchasing,...,1,0,0.003079,0.026846,corporate / office,Engineering,0,일치,1.0,1
1,/PH-00/Philippines,AS,0.066667,23122,End-Customer,Enterprise,NaN,NaN,NaN,media and communication,...,1,0,0.003079,0.026846,corporate / office,Advertising,1,일치,1.5,1
2,/Kolkata /India,AS,0.088889,1755,End-Customer,Enterprise,NaN,NaN,NaN,engineering,...,1,0,0.003079,0.026846,corporate / office,Construction,2,일치,0.0,1
3,/Bhubaneswar/India,AS,0.088889,4919,End-Customer,Enterprise,NaN,NaN,NaN,entrepreneurship,...,1,0,0.003079,0.026846,corporate / office,IT/Software,3,일치,1.0,1
4,/Hyderabad/India,AS,0.088889,17126,Specifier/ Influencer,Enterprise,NaN,NaN,NaN,consulting,...,0,0,0.003079,0.026846,corporate / office,NaN,4,일치,2.5,1


In [52]:
x_train, x_val, y_train, y_val = train_test_split(
    df_train.drop("is_converted", axis=1),
    df_train["is_converted"],
    test_size=0.2,
    shuffle=True,
    random_state=42
)

In [53]:
from catboost import CatBoostClassifier, Pool

학습 및 검증 세팅

---
범주형 인덱스 설정: cat_features=[]
자연어 텍스트 임베딩 설정: embedding_features=[]


In [54]:
#비정형 데이터 결측치 처리
text_features=[0,13,16]

for idx in text_features:
    x_train.iloc[:, idx].fillna('', inplace=True)
    x_val.iloc[:, idx].fillna('', inplace=True)
    
cat_features=[1,4,5,9,10,11,12,14,15,21,22,23,24]

for idx in cat_features:
    x_train.iloc[:, idx].fillna('none', inplace=True)
    x_val.iloc[:, idx].fillna('none', inplace=True)
    
nantozero=[6,7,8]
for idx in nantozero:
    x_train.iloc[:, idx].fillna(0, inplace=True)
    x_val.iloc[:, idx].fillna(0, inplace=True)

# 학습데이터 세팅
train_data = Pool(
    x_train,
    label = y_train,
    cat_features=[1,4,5,9,10,11,12,14,15,21,22,23,24],
    text_features= text_features
)

# 검증데이터 세팅
eval_data = Pool(
    x_val,
    label = y_val,
    cat_features=[1,4,5,9,10,11,12,14,15,21,22,23,24],
    text_features= text_features
)

In [55]:
# 모델 정의
model = CatBoostClassifier(
    iterations=2000,  # 더 많은 반복으로 모델이 데이터에서 패턴을 더 잘 학습하도록 합니다.
    learning_rate=0.05,  # 학습률을 증가시켜 학습 속도를 높입니다. 너무 높으면 과적합의 위험이 있으므로 조심하세요.
    depth=6,  # 트리의 깊이를 설정합니다. 너무 깊은 트리는 과적합을 일으킬 수 있습니다.
    l2_leaf_reg=3,  # L2 정규화 계수를 사용하여 모델의 과적합을 방지합니다.
    eval_metric='AUC',  # 이진 분류 작업에 적합한 평가 지표인 AUC를 사용합니다.
    border_count=254,  # 기본값인 254를 사용하거나, 모델의 성능을 개선하기 위해 조정할 수 있습니다.
    random_seed=42,  # 결과의 재현성을 위해 랜덤 시드를 설정합니다.
    verbose=100,  # 학습 과정에서 100번의 반복마다 메트릭을 출력합니다.
    use_best_model=True,  # 검증 세트에서 가장 좋은 모델을 사용합니다.
    od_type='Iter',  # 반복에 대한 과적합 감지 유형을 설정합니다.
    od_wait=50,# 지정된 반복 동안 성능이 개선되지 않으면 학습을 중단합니다.
    class_weights= [1,11.5] # 1:10 일때 현재 성능 최고
)

# 모델 학습 및 검증
model.fit(train_data,
          eval_set= eval_data) # 학습과정 모니터링 설정



print(model.get_best_score())

0:	test: 0.8880520	best: 0.8880520 (0)	total: 351ms	remaining: 11m 42s
100:	test: 0.9806222	best: 0.9806222 (100)	total: 19.2s	remaining: 6m 1s
200:	test: 0.9822928	best: 0.9822928 (200)	total: 35s	remaining: 5m 13s
300:	test: 0.9839410	best: 0.9839410 (300)	total: 51.8s	remaining: 4m 52s
400:	test: 0.9845897	best: 0.9845983 (395)	total: 1m 8s	remaining: 4m 32s
500:	test: 0.9850151	best: 0.9850564 (491)	total: 1m 24s	remaining: 4m 13s
600:	test: 0.9852569	best: 0.9852569 (600)	total: 1m 40s	remaining: 3m 54s
700:	test: 0.9853252	best: 0.9853456 (677)	total: 1m 56s	remaining: 3m 36s
800:	test: 0.9855352	best: 0.9855574 (793)	total: 2m 13s	remaining: 3m 19s
900:	test: 0.9856497	best: 0.9856497 (900)	total: 2m 29s	remaining: 3m 2s
1000:	test: 0.9858266	best: 0.9858280 (996)	total: 2m 45s	remaining: 2m 45s
1100:	test: 0.9860005	best: 0.9860005 (1100)	total: 3m 1s	remaining: 2m 28s
1200:	test: 0.9860733	best: 0.9860733 (1200)	total: 3m 17s	remaining: 2m 11s
1300:	test: 0.9861348	best: 0.986

##검증 데이터로 평가지표 출력해보기

In [56]:
def get__clf_eval(y_test, y_pred=None):
    confusion = confusion_matrix(y_test, y_pred, labels=[True, False]) #(답, 예측)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, labels=[True, False])
    recall = recall_score(y_test, y_pred)
    F1 = f1_score(y_test, y_pred, labels=[True, False])

    print("오차행렬:\n", confusion)
    print("\n정확도: {:.4f}".format(accuracy))
    print("정밀도: {:.4f}".format(precision))
    print("재현율: {:.4f}".format(recall))
    print("F1: {:.4f}".format(F1))

In [57]:
pred = model.predict(x_val)
pred = np.where(pred == 1, True, False)
get__clf_eval(y_val, pred)

오차행렬:
 [[  912    73]
 [  487 10388]]

정확도: 0.9528
정밀도: 0.6519
재현율: 0.9259
F1: 0.7651


##예측

In [58]:
# 예측에 필요한 데이터 분리
x_test = df_test.drop(["is_converted", "id"], axis=1)

In [59]:
#가공변수+열삭제
x_test['same_country'] = (x_test['customer_country'] == x_test['customer_country.1']).astype(int)
x_test['same_country'] = x_test['same_country'].map({0: '불일치', 1: '일치'})
x_test.drop('customer_country.1', axis=1, inplace=True)

In [60]:
# lead_desc_length의 전체 평균 계산
lead_desc_length_avg = x_test['lead_desc_length'].mean()

In [61]:
x_test['historical_existing_cnt'].fillna(0, inplace=True)

In [62]:
x_test['engagement_weight_sum'] = x_test.apply(calculate_engagement_weight, axis=1)

In [63]:
x_test.drop('bant_submit', axis=1, inplace=True)

In [64]:
x_test.drop('lead_desc_length', axis=1, inplace=True)

In [65]:
x_test.drop('historical_existing_cnt', axis=1, inplace=True)

In [66]:
#######
nantozero=[6,7,8]
for idx in nantozero:
    x_test.iloc[:, idx].fillna(0, inplace=True)
    
text_features=[0,13,16]

for idx in text_features:
    x_test.iloc[:, idx].fillna('', inplace=True)

cat_features=[1,4,5,9,10,11,12,14,15,21,22,23,24]

for idx in cat_features:
    x_test.iloc[:, idx].fillna('none', inplace=True)    

#테스트 데이터 세팅
test_data = Pool(
    x_test,
    cat_features=[1,4,5,9,10,11,12,14,15,21,22,23,24],
    text_features= text_features
)

In [67]:
test_pred = model.predict(x_test)
#sum(test_pred) # True로 예측된 개수

In [68]:
# 1,0 을 True,False로 바꿔주기
test_pred = np.where(test_pred == 1, True, False)
test_pred

array([False,  True,  True, ..., False, False,  True])

##제출 파일 저장

In [69]:
# 제출 데이터 읽어오기 (df_test는 전처리된 데이터가 저장됨)
df_sub = pd.read_csv("submission.csv")
df_sub["is_converted"] = test_pred

# 제출 파일 저장
df_sub.to_csv("submission.csv", index=False)